# Image Generation Using Replicate's Stable Diffusion

This script generates AI images based on haunted place descriptions using Replicate’s Stable Diffusion model. It loads the haunted places dataset, constructs prompts from the “Description” column, and uses the Replicate API to create one illustration per haunted place. I used it to generate images for haunted places with IDs 0–1999 and 4000–7999, adjusting the rows in batches depending on which images I wanted to produce.

In [190]:
import replicate
import requests
import os
import pandas as pd

# Set Replicate API token (I pasted my actual token to run this code but am not supposed to share it)
os.environ["REPLICATE_API_TOKEN"] = "token"

# Load dataset
df = pd.read_csv("../data/processed/haunted_places_features_added_v2.tab", sep="\t")

# Use fixed existing path
output_dir = "../data/generated_images"

# Prompt generator
def make_prompt(desc):
    return f"Haunted place illustration: {desc}"

# Image generator using Replicate
def generate_sd_image(prompt, image_path):
    try:
        output = replicate.run(
            "stability-ai/stable-diffusion:ac732df83cea7fff18b8472768c88ad041fa750ff7682a21affe81863cbe77e4",
            input={
                "width": 768,
                "height": 768,
                "prompt": prompt,
                "scheduler": "K_EULER",
                "num_outputs": 1,
                "guidance_scale": 7.5,
                "num_inference_steps": 50
            }
        )
        image_url = output[0]
        img_data = requests.get(image_url).content
        with open(image_path, 'wb') as f:
            f.write(img_data)
        return image_path
    except Exception as e:
        print(f"Failed to generate: {e}")
        return None

# Generate images
for idx, row in df.iloc[4000:6000].iterrows():  # I adjusted range depending on haunted places I wanted to produce images for
    haunted_id = str(row["Haunted_Places_Id"])
    prompt = make_prompt(row["Description"])

    filename = f"hpimg_{haunted_id}.png"
    image_path = os.path.join(output_dir, filename)  # absolute path
    relative_path = os.path.relpath(image_path, start=".")  # relative for TSV

    path = generate_sd_image(prompt, image_path)
    if path:
        df.at[idx, "ai_image_path"] = relative_path

Failed to generate: NSFW content detected. Try running it again, or try a different prompt.
Failed to generate: NSFW content detected. Try running it again, or try a different prompt.
